# 3.31 — Online & Incremental Learning

Online and incremental learning train a model by updating it as examples arrive instead of waiting for one fixed full batch. In this lesson, we build the update rule, the streaming loss average, the cost-aware selection score, the validation gap, and the stability knobs from scratch with NumPy so every number can be inspected.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build online learning one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the update, score, and final model choice are never a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on variables so it never clashes with examples below.

In [ ]:
import numpy as np  # arrays, dot products, gradients, and deterministic toy streams.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for the toy stream.

▶ What you'll see: no printed output; this cell prepares NumPy and Matplotlib for the walkthrough.

### 1. A data stream and one prediction rule

Online learning begins with a stream: example 1 arrives, then example 2, then example 3. We do not solve one giant normal equation; we keep a current weight vector `w` and score each new example by a dot product. For a tiny regression model, the prediction is \(\hat y_t=w_t^\top x_t\), and the loss tells us how badly the current weights behave on the current arrival.

In [ ]:
X_w = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])  # bias feature plus one signal feature.
y_w = np.array([0.669, 0.956, 2.185])  # three incoming targets chosen for hand-checkable losses.
w0_w = np.array([0.0, 0.0])  # start with no prior slope or intercept.

print("stream shape:", X_w.shape)  # three examples, two parameters.
print("initial weights:", w0_w)  # the current model before the first example.

▶ What you'll see: a 3×2 stream and a zero weight vector, so the first predictions are initially zero.

In [ ]:
pred0_w = X_w @ w0_w  # score every example with the starting model just for inspection.
loss0_w = 0.5 * (pred0_w - y_w) ** 2  # half squared loss per example.

print("initial predictions:", np.round(pred0_w, 3))  # all zeros before learning.
print("initial losses:", np.round(loss0_w, 3))  # large because the model has not adapted.

plt.figure(figsize=(4.6, 3))
plt.bar(["ex1", "ex2", "ex3"], loss0_w, color="crimson")
plt.title("1: losses before any online updates")
plt.ylabel("0.5(pred-y)^2")
plt.show()

▶ What you'll see: all predictions start at 0, so the loss bars reflect the target magnitudes.

*Why it's done this way:* a stream forces the model to expose its current beliefs before seeing future examples. The dot product is linear enough that its gradient is simple, but expressive enough to show the core online-learning contract: predict now, pay a loss now, update for the next arrival.

### 2. The online gradient update

The core rule is \(w_{t+1}=w_t-\eta_t\nabla\ell_t(w_t)\). With half squared error \(\ell_t(w)=\frac12(w^\top x_t-y_t)^2\), the gradient is \((w^\top x_t-y_t)x_t\). If the prediction is too small, the residual is negative, so subtracting the gradient pushes `w` in the direction of `x_t` and raises similar future predictions.

In [ ]:
x1_w = X_w[0]  # first arriving example.
y1_w = y_w[0]  # first target.
eta_w = 0.10  # fixed step size for the first demonstration.
pred1_before_w = float(w0_w @ x1_w)  # current prediction before learning from example 1.
grad1_w = (pred1_before_w - y1_w) * x1_w  # gradient of half squared loss.

print("prediction before:", round(pred1_before_w, 3))
print("gradient:", np.round(grad1_w, 3))

assert round(float(0.5 * (pred1_before_w - y1_w) ** 2), 3) == 0.224

▶ What you'll see: the first verified loss is 0.224 and the gradient points opposite the target feature direction.

In [ ]:
w1_w = w0_w - eta_w * grad1_w  # one online update after the first example.
pred1_after_w = float(w1_w @ x1_w)  # re-score the same example after the correction.

print("updated weights:", np.round(w1_w, 4))
print("prediction after:", round(pred1_after_w, 3))

assert np.allclose(np.round(w1_w, 4), [0.0669, 0.0])
plt.figure(figsize=(4.4, 3))
plt.bar(["before", "after", "target"], [pred1_before_w, pred1_after_w, y1_w], color=["gray", "teal", "black"])
plt.title("2: one online update moves toward the target")
plt.ylabel("value")
plt.show()

▶ What you'll see: the prediction moves from 0 toward 0.669 after exactly one gradient step.

*Why it's done this way:* gradient descent moves opposite the direction that most increases loss. In online learning we apply that correction immediately, so the next example sees a slightly better model without waiting for a full dataset pass.

### 3. Empirical risk as a running average

The lesson's raw training score is an empirical-risk average. For the toy instance the verified per-example losses are 0.224, 0.148, and 0.488, so \(R_S=(0.224+0.148+0.488)/3=0.287\). Online learners often maintain this quantity incrementally instead of recomputing the full mean from scratch.

In [ ]:
losses_w = np.array([0.224, 0.148, 0.488])  # verified per-example losses from the lesson text.
risk_w = float(np.mean(losses_w))  # empirical risk = average loss over observed examples.

print("losses:", losses_w)
print("empirical risk R_S:", round(risk_w, 3))

assert round(risk_w, 3) == 0.287

▶ What you'll see: the raw empirical-risk score is 0.287.

In [ ]:
running_w = []  # running averages after each arrival.
total_w = 0.0
for t_w, loss_w in enumerate(losses_w, start=1):
    total_w += loss_w  # add only the new loss.
    running_w.append(total_w / t_w)  # divide by examples seen so far.

print("running averages:", np.round(running_w, 3))

plt.figure(figsize=(4.6, 3))
plt.plot([1, 2, 3], running_w, marker="o", color="purple")
plt.axhline(risk_w, linestyle="--", color="gray", label="final R_S")
plt.title("3: empirical risk as a streaming average")
plt.xlabel("examples seen")
plt.ylabel("average loss")
plt.legend()
plt.show()

▶ What you'll see: the average changes after each arrival and lands at the same 0.287 you get from the batch mean.

*Why it's done this way:* empirical risk is an average because we want one comparable number that is not inflated merely by having more examples. The streaming formula preserves that scale while using constant memory: keep the count and total, then update when a new loss arrives.

### 4. Adding cost or regularization to the decision score

A low raw loss is not always the best choice. The lesson adds a complexity, regularization, or operational cost of 0.060, giving \(score=R_S+cost=0.287+0.060=0.347\). This is the model-selection score, not just a decorative add-on.

In [ ]:
cost_w = 0.060  # stabilizing or operational cost from the lesson text.
score_w = risk_w + cost_w  # selection score combines fit and cost.

print("raw risk:", round(risk_w, 3))
print("cost:", round(cost_w, 3))
print("decision score:", round(score_w, 3))

assert round(score_w, 3) == 0.347

▶ What you'll see: the score used for selection is 0.347, larger than the raw training average.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["R_S", "cost", "R_S + cost"], [risk_w, cost_w, score_w], color=["teal", "orange", "black"])
plt.title("4: fit plus cost defines the decision score")
plt.ylabel("score component")
plt.show()

▶ What you'll see: the final bar is the sum of the raw fit and the stabilizing cost.

*Why it's done this way:* the cost term is a guardrail against chasing the prettiest training fragment. Mathematically, adding it changes the objective's scale, so the selected model must be the one with the lowest full score, not necessarily the lowest raw loss.

### 5. Comparing alternatives with absolute and relative gaps

A more flexible alternative reaches a decision score of 0.387. The baseline score 0.347 is lower by 0.040, and the relative gap is \((0.387-0.347)/0.387=0.103\). The relative gap matters because a tiny absolute improvement can vanish under sampling noise.

In [ ]:
score_cmp_w = 0.347  # use the rounded decision score reported by the lesson arithmetic.
alt_score_w = 0.387  # tempting flexible alternative.
gap_w = alt_score_w - score_cmp_w  # absolute evidence for preferring the lower score.
rel_gap_w = gap_w / alt_score_w  # scale-aware gap.

print("baseline score:", round(score_cmp_w, 3), "alternative:", round(alt_score_w, 3))
print("gap:", round(gap_w, 3), "relative gap:", round(rel_gap_w, 3))

assert round(gap_w, 3) == 0.040
assert round(rel_gap_w, 3) == 0.103

▶ What you'll see: the baseline wins by 0.040, about 10.3% of the alternative's score.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["baseline", "flexible alt"], [score_cmp_w, alt_score_w], color=["seagreen", "crimson"])
plt.title("5: lower full score wins the comparison")
plt.ylabel("decision score")
plt.show()

▶ What you'll see: the flexible alternative has the taller bar, so it loses under the lesson's score scale.

*Why it's done this way:* model comparison is meaningful only after both candidates are on the same score scale. The absolute gap gives the raw margin; the relative gap asks whether that margin is large enough to take seriously.

### 6. Stability knobs: decay, mini-batches, and final choice

Online learning can be noisy because one example can pull the model sharply. A stability knob — regularization, averaging, a decaying step size, or a smaller incremental update — deliberately gives up brittle variation. In the lesson's verified arithmetic, a 20% reduction turns 0.347 into \(0.80\cdot0.347=0.278\), and the final decision is \(\min(0.347,0.387,0.278)=0.278\).

In [ ]:
stable_w = 0.80 * score_cmp_w  # stabilizing knob reduces the rounded score by 20%.
choices_w = np.array([score_cmp_w, alt_score_w, stable_w])
labels_w = ["baseline", "flexible", "stabilized"]
best_idx_w = int(np.argmin(choices_w))

print("stabilized score:", round(stable_w, 3))
print("best choice:", labels_w[best_idx_w], round(float(choices_w[best_idx_w]), 3))

assert round(stable_w, 3) == 0.278
assert round(float(np.min(choices_w)), 3) == 0.278

▶ What you'll see: the stabilized version has the lowest verified decision score.

In [ ]:
etas_w = 0.3 / np.sqrt(np.arange(1, 11))  # a common decaying learning-rate schedule.

print("first three learning rates:", np.round(etas_w[:3], 3))

plt.figure(figsize=(4.6, 3))
plt.plot(np.arange(1, 11), etas_w, marker="o", color="navy")
plt.title("6: decaying η_t makes later updates calmer")
plt.xlabel("time step t")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: early steps are large for quick adaptation, while later steps shrink for stability.

*Why it's done this way:* stability is the bargain between adaptability and variance. Early updates let the model move when it knows little; later smaller updates prevent one noisy arrival from overwriting accumulated evidence.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · A stream makes one prediction at a time

An online learner scores each arriving row with the current weights before it updates on later rows.

In [ ]:
import numpy as np                              # arrays, dot products, and losses.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t1_X = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0], [1.0, 3.0], [1.0, 4.0], [1.0, 5.0]])
t1_y = np.array([0.5, 1.0, 1.5, 2.1, 2.4, 3.0])
t1_w = np.array([0.4, 0.45])
t1_pred = t1_X @ t1_w                           # -> [0.4, 0.85, 1.3, 1.75, 2.2, 2.65]
t1_residual = t1_pred - t1_y                    # -> [-0.1, -0.15, -0.2, -0.35, -0.2, -0.35]
t1_loss = 0.5 * t1_residual ** 2                # -> [0.005, 0.0112, 0.02, 0.0613, 0.02, 0.0613]

print("stream X:", t1_X.tolist())              # -> [[1.0, 0.0], [1.0, 1.0], [1.0, 2.0], [1.0, 3.0], [1.0, 4.0], [1.0, 5.0]]
print("targets:", t1_y.tolist())               # -> [0.5, 1.0, 1.5, 2.1, 2.4, 3.0]
print("weights:", t1_w.tolist())               # -> [0.4, 0.45]
print("predictions:", np.round(t1_pred, 3).tolist())  # -> [0.4, 0.85, 1.3, 1.75, 2.2, 2.65]
print("residuals:", np.round(t1_residual, 3).tolist())  # -> [-0.1, -0.15, -0.2, -0.35, -0.2, -0.35]
print("losses:", np.round(t1_loss, 4).tolist())  # -> [0.005, 0.0112, 0.02, 0.0613, 0.02, 0.0613]

assert round(float(t1_loss.mean()), 4) == 0.0298

plt.figure(figsize=(4.6, 3.0))
plt.plot(t1_X[:, 1], t1_y, "o-", label="target")
plt.plot(t1_X[:, 1], t1_pred, "x--", label="current prediction")
plt.xlabel("arrival feature")
plt.ylabel("value")
plt.legend()
plt.title("Toy 1 · current model scores the stream")
plt.show()

▶ What you'll see: the current model trails the targets, so every residual is negative.

### ✍️ Toy 2 · One gradient points opposite the residual

For half-squared loss, the gradient on one arriving example is `(prediction - target) * x`.

In [ ]:
import numpy as np                              # vector gradients.

t2_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t2_x = np.array([1.0, 2.0])
t2_y = 3.0
t2_w = np.array([0.5, 0.25])
t2_pred = float(t2_w @ t2_x)                   # -> 1.0
t2_residual = t2_pred - t2_y                   # -> -2.0
t2_loss = 0.5 * t2_residual ** 2               # -> 2.0
t2_grad = t2_residual * t2_x                   # -> [-2.0, -4.0]

print("x:", t2_x.tolist())                    # -> [1.0, 2.0]
print("y:", t2_y)                             # -> 3.0
print("w before:", t2_w.tolist())              # -> [0.5, 0.25]
print("prediction:", t2_pred)                 # -> 1.0
print("residual:", t2_residual)               # -> -2.0
print("loss:", t2_loss)                       # -> 2.0
print("gradient:", t2_grad.tolist())          # -> [-2.0, -4.0]

assert np.array_equal(t2_grad, np.array([-2.0, -4.0]))

plt.figure(figsize=(4.2, 3.0))
plt.quiver([0], [0], [t2_grad[0]], [t2_grad[1]], angles="xy", scale_units="xy", scale=1, color="crimson")
plt.xlim(-5, 1)
plt.ylim(-5, 1)
plt.axhline(0, color="gray", linewidth=0.7)
plt.axvline(0, color="gray", linewidth=0.7)
plt.title("Toy 2 · gradient direction")
plt.xlabel("bias component")
plt.ylabel("slope component")
plt.show()

▶ What you'll see: the gradient points negative because the prediction is too small.

### ✍️ Toy 3 · Updating immediately changes the next score

Subtracting the gradient moves the weights before the next example arrives.

In [ ]:
import numpy as np                              # online weight update.

t3_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t3_x = np.array([1.0, 2.0])
t3_y = 3.0
t3_w = np.array([0.5, 0.25])
t3_eta = 0.1
t3_pred_before = float(t3_w @ t3_x)            # -> 1.0
t3_grad = (t3_pred_before - t3_y) * t3_x       # -> [-2.0, -4.0]
t3_step = t3_eta * t3_grad                     # -> [-0.2, -0.4]
t3_w_new = t3_w - t3_step                      # -> [0.7, 0.65]
t3_pred_after = float(t3_w_new @ t3_x)         # -> 2.0
t3_loss_after = 0.5 * (t3_pred_after - t3_y) ** 2  # -> 0.5

print("eta:", t3_eta)                         # -> 0.1
print("prediction before:", t3_pred_before)   # -> 1.0
print("gradient:", t3_grad.tolist())          # -> [-2.0, -4.0]
print("eta * gradient:", t3_step.tolist())    # -> [-0.2, -0.4]
print("updated weights:", t3_w_new.tolist())  # -> [0.7, 0.65]
print("prediction after:", t3_pred_after)     # -> 2.0
print("loss after:", t3_loss_after)           # -> 0.5

assert np.allclose(t3_w_new, np.array([0.7, 0.65]))

plt.figure(figsize=(4.4, 2.8))
plt.bar(["before", "after", "target"], [t3_pred_before, t3_pred_after, t3_y], color=["gray", "teal", "black"])
plt.ylabel("prediction")
plt.title("Toy 3 · one update moves toward y")
plt.show()

▶ What you'll see: the prediction jumps from `1.0` to `2.0`, closer to the target `3.0`.

### ✍️ Toy 4 · Running risk updates by count and total

A streaming mean keeps only the running total and the number of examples seen.

In [ ]:
import numpy as np                              # cumulative sums and means.

t4_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t4_losses = np.array([0.5, 0.125, 0.32, 0.08, 0.18, 0.045])
t4_cumsum = np.cumsum(t4_losses)               # -> [0.5, 0.625, 0.945, 1.025, 1.205, 1.25]
t4_counts = np.arange(1, t4_losses.size + 1)   # -> [1, 2, 3, 4, 5, 6]
t4_running = t4_cumsum / t4_counts             # -> [0.5, 0.3125, 0.315, 0.2563, 0.241, 0.2083]
t4_final = float(t4_running[-1])               # -> 0.20833333333333334

print("losses:", t4_losses.tolist())           # -> [0.5, 0.125, 0.32, 0.08, 0.18, 0.045]
print("cumulative totals:", np.round(t4_cumsum, 3).tolist())  # -> [0.5, 0.625, 0.945, 1.025, 1.205, 1.25]
print("counts:", t4_counts.tolist())           # -> [1, 2, 3, 4, 5, 6]
print("running means:", np.round(t4_running, 4).tolist())  # -> [0.5, 0.3125, 0.315, 0.2563, 0.241, 0.2083]
print("final mean:", round(t4_final, 4))       # -> 0.2083

assert round(t4_final, 4) == 0.2083

plt.figure(figsize=(4.6, 3.0))
plt.plot(t4_counts, t4_running, marker="o", color="purple")
plt.axhline(t4_final, color="black", linestyle="--", label="final mean")
plt.xlabel("examples seen")
plt.ylabel("running risk")
plt.legend()
plt.title("Toy 4 · streaming empirical risk")
plt.show()

▶ What you'll see: the running average moves after each loss and settles at `0.2083`.

### ✍️ Toy 5 · Cost turns raw fit into a decision score

A regularization or operations cost is added to the raw fit before model selection.

In [ ]:
import numpy as np                              # norms and scalar scores.

t5_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t5_risk = 0.20833333333333334
t5_w = np.array([0.7, 0.4])
t5_lambda = 0.2
t5_norm_sq = float(t5_w @ t5_w)                # -> 0.6499999999999999
t5_cost = 0.5 * t5_lambda * t5_norm_sq         # -> 0.06499999999999999
t5_score = t5_risk + t5_cost                   # -> 0.2733333333333333

print("risk:", round(t5_risk, 4))              # -> 0.2083
print("weights:", t5_w.tolist())              # -> [0.7, 0.4]
print("||w||^2:", round(t5_norm_sq, 3))        # -> 0.65
print("cost:", round(t5_cost, 3))              # -> 0.065
print("decision score:", round(t5_score, 4))   # -> 0.2733

assert round(t5_score, 4) == 0.2733

plt.figure(figsize=(4.4, 2.8))
plt.bar(["risk", "cost", "score"], [t5_risk, t5_cost, t5_score], color=["teal", "orange", "black"])
plt.ylabel("value")
plt.title("Toy 5 · fit plus cost")
plt.show()

▶ What you'll see: the final score bar is the sum of the risk and cost bars.

### ✍️ Toy 6 · Absolute and relative gaps compare candidates

The absolute gap gives raw evidence; the relative gap scales that evidence by the alternative score.

In [ ]:
import numpy as np                              # scalar comparisons.

t6_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t6_baseline = 0.2733333333333333
t6_alternative = 0.315
t6_gap = t6_alternative - t6_baseline          # -> 0.041666666666666685
t6_relative_gap = t6_gap / t6_alternative      # -> 0.13227513227513232
t6_scores = np.array([t6_baseline, t6_alternative])

print("baseline:", round(t6_baseline, 4))      # -> 0.2733
print("alternative:", round(t6_alternative, 4))  # -> 0.315
print("absolute gap:", round(t6_gap, 4))       # -> 0.0417
print("relative gap:", round(t6_relative_gap, 4))  # -> 0.1323

assert round(t6_relative_gap, 4) == 0.1323

plt.figure(figsize=(4.2, 2.8))
plt.bar(["baseline", "alternative"], t6_scores, color=["seagreen", "crimson"])
plt.ylabel("decision score")
plt.title("Toy 6 · lower score wins by a gap")
plt.show()

▶ What you'll see: the baseline bar is lower by about 13.2% of the alternative score.

### ✍️ Toy 7 · Decay shrinks later update lengths

A decaying learning rate makes updates calmer as more examples have been seen.

In [ ]:
import numpy as np                              # square roots and elementwise products.

t7_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t7_steps = np.arange(1, 7)
t7_eta0 = 0.4
t7_etas = t7_eta0 / np.sqrt(t7_steps)           # -> [0.4, 0.2828, 0.2309, 0.2, 0.1789, 0.1633]
t7_grad_norms = np.array([2.0, 1.5, 1.2, 1.0, 0.8, 0.7])
t7_step_lengths = t7_etas * t7_grad_norms      # -> [0.8, 0.4243, 0.2771, 0.2, 0.1431, 0.1143]

print("steps:", t7_steps.tolist())             # -> [1, 2, 3, 4, 5, 6]
print("etas:", np.round(t7_etas, 4).tolist())  # -> [0.4, 0.2828, 0.2309, 0.2, 0.1789, 0.1633]
print("gradient norms:", t7_grad_norms.tolist())  # -> [2.0, 1.5, 1.2, 1.0, 0.8, 0.7]
print("step lengths:", np.round(t7_step_lengths, 4).tolist())  # -> [0.8, 0.4243, 0.2771, 0.2, 0.1431, 0.1143]

assert np.all(np.diff(t7_etas) < 0.0)

plt.figure(figsize=(4.6, 3.0))
plt.plot(t7_steps, t7_etas, marker="o", label="eta")
plt.plot(t7_steps, t7_step_lengths, marker="x", label="eta · ||grad||")
plt.xlabel("time step")
plt.ylabel("size")
plt.legend()
plt.title("Toy 7 · decay stabilizes updates")
plt.show()

▶ What you'll see: both the learning rate and the resulting step lengths shrink over time.

### ✍️ Toy 8 · Mini-batches average gradients before moving

A mini-batch update computes several per-example gradients, averages them, and then moves once.

In [ ]:
import numpy as np                              # batch gradients.

t8_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t8_X = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0], [1.0, 3.0]])
t8_y = np.array([0.4, 1.1, 1.6, 2.2])
t8_w = np.array([0.3, 0.4])
t8_eta = 0.2
t8_pred = t8_X @ t8_w                          # -> [0.3, 0.7, 1.1, 1.5]
t8_residual = t8_pred - t8_y                   # -> [-0.1, -0.4, -0.5, -0.7]
t8_grads = t8_residual[:, None] * t8_X         # -> [[-0.1, -0.0], [-0.4, -0.4], [-0.5, -1.0], [-0.7, -2.1]]
t8_batch_grad = t8_grads.mean(axis=0)          # -> [-0.425, -0.875]
t8_w_new = t8_w - t8_eta * t8_batch_grad       # -> [0.385, 0.575]

print("predictions:", np.round(t8_pred, 3).tolist())  # -> [0.3, 0.7, 1.1, 1.5]
print("residuals:", np.round(t8_residual, 3).tolist())  # -> [-0.1, -0.4, -0.5, -0.7]
print("per-example gradients:", np.round(t8_grads, 3).tolist())  # -> [[-0.1, -0.0], [-0.4, -0.4], [-0.5, -1.0], [-0.7, -2.1]]
print("batch gradient:", np.round(t8_batch_grad, 3).tolist())  # -> [-0.425, -0.875]
print("updated weights:", np.round(t8_w_new, 3).tolist())  # -> [0.385, 0.575]

assert np.allclose(np.round(t8_w_new, 3), np.array([0.385, 0.575]))

plt.figure(figsize=(4.4, 3.0))
plt.quiver([0], [0], [t8_batch_grad[0]], [t8_batch_grad[1]], angles="xy", scale_units="xy", scale=1, color="crimson")
plt.xlim(-1.0, 0.2)
plt.ylim(-1.0, 0.2)
plt.axhline(0, color="gray", linewidth=0.7)
plt.axvline(0, color="gray", linewidth=0.7)
plt.title("Toy 8 · average gradient")
plt.show()

▶ What you'll see: the batch gradient points down-left, so subtracting it increases both weights.

### ✍️ Toy 9 · Final selection chooses the lowest full score

After all stability knobs are scored on the same scale, selection is an `argmin` over full decision
scores.

In [ ]:
import numpy as np                              # argmin over candidate scores.

t9_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t9_labels = np.array(["baseline", "flexible", "stabilized", "mini-batch"])
t9_scores = np.array([0.2733, 0.3150, 0.2510, 0.2890])
t9_best = int(np.argmin(t9_scores))            # -> 2
t9_best_label = str(t9_labels[t9_best])        # -> stabilized
t9_best_score = float(t9_scores[t9_best])      # -> 0.251

print("labels:", t9_labels.tolist())           # -> ['baseline', 'flexible', 'stabilized', 'mini-batch']
print("scores:", t9_scores.tolist())           # -> [0.2733, 0.315, 0.251, 0.289]
print("best index:", t9_best)                 # -> 2
print("best label:", t9_best_label)           # -> stabilized
print("best score:", t9_best_score)           # -> 0.251

assert t9_best_label == "stabilized"

plt.figure(figsize=(4.8, 2.8))
plt.bar(t9_labels, t9_scores, color=["gray", "crimson", "teal", "orange"])
plt.ylabel("decision score")
plt.xticks(rotation=15)
plt.title("Toy 9 · choose the minimum")
plt.show()

▶ What you'll see: the stabilized bar is the lowest, so it is selected.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for vectors, losses, gradients, and streaming averages.
import matplotlib.pyplot as plt # load Matplotlib for compact plots that inspect learning behavior.
np.random.seed(0) # make every stochastic example reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Create a tiny stream

**Goal.** Build three arriving examples, because online learning is organized by time rather than by one static table. We build it in 2 steps.

In [ ]:
X_b1 = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]]) # store bias plus one feature for three arrivals.
y_b1 = np.array([0.669, 0.956, 2.185]) # store the target that arrives with each feature row.

print("X_b1 shape:", X_b1.shape) # inspect examples by features.
print("y_b1:", y_b1) # inspect the incoming targets.

▶ What you'll see: three examples are ready to be processed in order.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact target plot.
plt.plot([1, 2, 3], y_b1, marker="o", color="teal") # draw target value by arrival time.
plt.title("Basic 1: target stream") # title the plot.
plt.xlabel("arrival t") # label time steps.
plt.ylabel("target y") # label target scale.
plt.show() # display the plot.

▶ What you'll see: the third target is largest, so a fixed zero model would miss it most.

👀 Takeaway: online learning treats examples as an ordered stream of prediction-and-update events.

### Basic 2 — Predict with current weights

**Goal.** Use a dot product to score the current example, because the online update needs a prediction before it can compute a loss. We build it in 2 steps.

In [ ]:
x_b2 = np.array([1.0, 2.0]) # define one example with bias and feature value 2.
w_b2 = np.array([0.2, 0.7]) # define current intercept and slope.
pred_b2 = float(w_b2 @ x_b2) # compute w dot x.

print("prediction:", round(pred_b2, 3)) # inspect the score.

assert round(pred_b2, 3) == 1.6 # verify 0.2 + 0.7*2.

▶ What you'll see: the current model predicts 1.6 for this example.

In [ ]:
plt.figure(figsize=(4, 3)) # create a contribution chart.
plt.bar(["bias", "feature"], w_b2 * x_b2, color="purple") # show terms that sum to the prediction.
plt.title("Basic 2: dot-product pieces") # title the chart.
plt.ylabel("contribution") # label contribution scale.
plt.show() # display the plot.

▶ What you'll see: the feature term contributes 1.4 and the bias contributes 0.2.

👀 Takeaway: the dot product exposes exactly which coordinates drive the current prediction.

### Basic 3 — Compute one half-squared loss

**Goal.** Measure one prediction error, because online gradients are derived from the current example's loss. We build it in 2 steps.

In [ ]:
pred_b3 = 0.0 # start with the initial model's first prediction.
y_b3 = 0.669 # use the first lesson target.
err_b3 = pred_b3 - y_b3 # compute prediction minus target.
loss_b3 = 0.5 * err_b3 ** 2 # compute half squared error.

print("error:", round(err_b3, 3), "loss:", round(loss_b3, 3)) # inspect the signed miss and penalty.

assert round(loss_b3, 3) == 0.224 # verify the lesson's first loss.

▶ What you'll see: the model underpredicts, and the verified loss is 0.224.

In [ ]:
pred_grid_b3 = np.linspace(-0.5, 1.5, 100) # sweep possible predictions.
loss_grid_b3 = 0.5 * (pred_grid_b3 - y_b3) ** 2 # compute loss curve.
plt.figure(figsize=(4, 3)) # create a loss-curve figure.
plt.plot(pred_grid_b3, loss_grid_b3, color="navy") # draw the half-squared-error bowl.
plt.scatter([pred_b3], [loss_b3], color="red") # mark the current prediction.
plt.title("Basic 3: half-squared loss") # title the curve.
plt.xlabel("prediction") # label prediction axis.
plt.ylabel("loss") # label loss axis.
plt.show() # display the plot.

▶ What you'll see: loss is minimized at the target and grows quadratically away from it.

👀 Takeaway: squared loss gives a smooth local objective for each arriving example.

### Basic 4 — Derive one gradient

**Goal.** Compute \((w^\top x-y)x\), because this is the direction that most increases the current half-squared loss. We build it in 2 steps.

In [ ]:
w_b4 = np.array([0.0, 0.0]) # define current weights.
x_b4 = np.array([1.0, 0.0]) # define the first arriving example.
y_b4 = 0.669 # define its target.
grad_b4 = (float(w_b4 @ x_b4) - y_b4) * x_b4 # compute gradient.

print("gradient:", np.round(grad_b4, 3)) # inspect derivative with respect to each weight.

assert np.allclose(np.round(grad_b4, 3), [-0.669, -0.0]) # verify the worked gradient.

▶ What you'll see: only the bias coordinate receives a gradient because the feature is zero.

In [ ]:
plt.figure(figsize=(4, 3)) # create a gradient bar chart.
plt.bar(["w0", "w1"], grad_b4, color="crimson") # show gradient entries.
plt.axhline(0, color="black", linewidth=1) # add zero baseline.
plt.title("Basic 4: gradient components") # title the plot.
plt.ylabel("gradient") # label derivative scale.
plt.show() # display the plot.

▶ What you'll see: the negative gradient means descent will increase the bias weight.

👀 Takeaway: gradients convert an error into coordinate-wise instructions for changing weights.

### Basic 5 — Take one online update

**Goal.** Apply \(w_{t+1}=w_t-\eta\nabla\ell_t(w_t)\), because online learning updates immediately after each example. We build it in 2 steps.

In [ ]:
w_b5 = np.array([0.0, 0.0]) # start from zero weights.
grad_b5 = np.array([-0.669, -0.0]) # reuse the first-example gradient.
eta_b5 = 0.10 # set the learning rate.
w_new_b5 = w_b5 - eta_b5 * grad_b5 # apply one online update.

print("new weights:", np.round(w_new_b5, 4)) # inspect updated parameters.

assert np.allclose(np.round(w_new_b5, 4), [0.0669, 0.0]) # verify the update.

▶ What you'll see: the bias moves from 0 to 0.0669.

In [ ]:
plt.figure(figsize=(4, 3)) # create a before/after chart.
plt.bar(["old w0", "new w0"], [w_b5[0], w_new_b5[0]], color=["gray", "teal"]) # compare bias weight.
plt.title("Basic 5: one online weight change") # title the chart.
plt.ylabel("weight value") # label weight scale.
plt.show() # display the plot.

▶ What you'll see: a small positive update caused by an underprediction.

👀 Takeaway: online updates are small local corrections, not full retraining steps.

### Basic 6 — Average per-example losses

**Goal.** Reproduce the empirical risk from the lesson's three verified losses, because raw training fit is an average. We build it in 2 steps.

In [ ]:
losses_b6 = np.array([0.224, 0.148, 0.488]) # store the verified per-example losses.
risk_b6 = float(np.mean(losses_b6)) # average them into empirical risk.

print("R_S:", round(risk_b6, 3)) # inspect the raw training score.

assert round(risk_b6, 3) == 0.287 # verify the lesson number.

▶ What you'll see: the empirical risk is 0.287.

In [ ]:
plt.figure(figsize=(4, 3)) # create a loss breakdown plot.
plt.bar(["l1", "l2", "l3"], losses_b6, color="orange") # show each loss term.
plt.axhline(risk_b6, color="black", linestyle="--", label="mean") # show the average.
plt.title("Basic 6: losses averaged into R_S") # title the chart.
plt.legend() # show average label.
plt.show() # display the plot.

▶ What you'll see: the mean line summarizes the three unequal loss bars.

👀 Takeaway: empirical risk is the normalized training quantity that online updates try to reduce.

### Basic 7 — Maintain a running mean

**Goal.** Update the empirical risk incrementally, because a stream may be too large to store forever. We build it in 2 steps.

In [ ]:
losses_b7 = np.array([0.224, 0.148, 0.488]) # define the arriving losses.
running_b7 = np.cumsum(losses_b7) / np.arange(1, 4) # compute running averages.

print("running means:", np.round(running_b7, 3)) # inspect average after each arrival.

assert round(float(running_b7[-1]), 3) == 0.287 # verify final risk matches the batch average.

▶ What you'll see: the running estimate changes as each loss arrives.

In [ ]:
plt.figure(figsize=(4, 3)) # create a streaming-average figure.
plt.plot([1, 2, 3], running_b7, marker="o", color="seagreen") # plot risk after each example.
plt.title("Basic 7: running empirical risk") # title the curve.
plt.xlabel("examples seen") # label x axis.
plt.ylabel("running average") # label y axis.
plt.show() # display the plot.

▶ What you'll see: the final point equals the full empirical risk without storing the whole history.

👀 Takeaway: incremental statistics are enough for many online monitoring signals.

### Basic 8 — Add the cost term

**Goal.** Build the decision score from raw risk plus cost, because selection should include the method's guardrail. We build it in 2 steps.

In [ ]:
risk_b8 = 0.287 # use the rounded empirical risk from the lesson.
cost_b8 = 0.060 # use the stated complexity or operational cost.
score_b8 = risk_b8 + cost_b8 # add fit and cost.

print("score:", round(score_b8, 3)) # inspect the selection score.

assert round(score_b8, 3) == 0.347 # verify the lesson score.

▶ What you'll see: the full decision score is 0.347.

In [ ]:
plt.figure(figsize=(4, 3)) # create a component plot.
plt.bar(["risk", "cost"], [risk_b8, cost_b8], color=["teal", "orange"]) # show additive pieces.
plt.title("Basic 8: score components") # title the chart.
plt.ylabel("value") # label scale.
plt.show() # display the plot.

▶ What you'll see: cost is smaller than risk but still changes the number being optimized.

👀 Takeaway: dropping the cost term silently changes the learning objective.

### Basic 9 — Compute the validation gap

**Goal.** Compare a baseline score with a flexible alternative, because online improvements must survive a fair score comparison. We build it in 2 steps.

In [ ]:
score_b9 = 0.347 # baseline decision score.
alt_b9 = 0.387 # more flexible alternative score.
gap_b9 = alt_b9 - score_b9 # absolute gap.
rel_b9 = gap_b9 / alt_b9 # relative gap on the alternative's scale.

print("gap:", round(gap_b9, 3), "relative:", round(rel_b9, 3)) # inspect both margins.

assert round(gap_b9, 3) == 0.040 # verify absolute gap.
assert round(rel_b9, 3) == 0.103 # verify relative gap.

▶ What you'll see: the baseline wins by 0.040, about 10.3% of the alternative.

In [ ]:
plt.figure(figsize=(4, 3)) # create a score comparison plot.
plt.bar(["baseline", "alternative"], [score_b9, alt_b9], color=["seagreen", "crimson"]) # compare model scores.
plt.title("Basic 9: score gap") # title the chart.
plt.ylabel("decision score") # label scale.
plt.show() # display the plot.

▶ What you'll see: the alternative bar is higher, so it is worse under a lower-is-better score.

👀 Takeaway: the validation or decision gap is the evidence for preferring one setting over another.

### Basic 10 — Choose the minimum score

**Goal.** Select among baseline, flexible, and stabilized scores, because the final decision uses the complete score implied by the method. We build it in 2 steps.

In [ ]:
scores_b10 = np.array([0.347, 0.387, 0.278]) # baseline, flexible alternative, stabilized version.
labels_b10 = np.array(["baseline", "flexible", "stabilized"]) # label each candidate.
best_b10 = int(np.argmin(scores_b10)) # find the lowest score.

print("best:", labels_b10[best_b10], round(float(scores_b10[best_b10]), 3)) # inspect chosen model.

assert round(float(np.min(scores_b10)), 3) == 0.278 # verify the final minimum.

▶ What you'll see: the stabilized version wins with score 0.278.

In [ ]:
plt.figure(figsize=(4.6, 3)) # create final decision plot.
plt.bar(labels_b10, scores_b10, color=["gray", "crimson", "teal"]) # compare all candidates.
plt.title("Basic 10: final score comparison") # title the chart.
plt.ylabel("score lower is better") # label scale.
plt.show() # display the plot.

▶ What you'll see: the stabilized bar is the shortest.

👀 Takeaway: the correct unit of judgment is the final score, not the prettiest training fragment.

## 🟡 Easy

### Easy 1 — Train a linear model online

**Goal.** Run several online updates on a toy stream, because incremental learning changes weights after each observed example. We build it in 4 steps.

In [ ]:
X_e1 = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0], [1.0, 3.0]]) # define four streamed examples.
y_e1 = np.array([0.5, 1.1, 1.9, 3.2]) # define targets from an approximately linear rule.
w_e1 = np.zeros(2) # initialize intercept and slope at zero.

print("initial w:", w_e1) # inspect starting parameters.

▶ What you'll see: the model starts with no intercept or slope.

In [ ]:
losses_e1 = [] # store loss after each online update.
for t_e1 in range(len(y_e1)): # process examples in stream order.
    pred_e1 = float(w_e1 @ X_e1[t_e1]) # predict before seeing the target update.
    err_e1 = pred_e1 - y_e1[t_e1] # compute residual.
    losses_e1.append(0.5 * err_e1 ** 2) # store current loss.
    w_e1 = w_e1 - 0.08 * err_e1 * X_e1[t_e1] # update weights using the current example only.

print("final w:", np.round(w_e1, 3)) # inspect learned parameters.
print("losses:", np.round(losses_e1, 3)) # inspect online losses.

▶ What you'll see: weights become positive and losses vary by arrival because learning is sequential.

In [ ]:
preds_e1 = X_e1 @ w_e1 # compute final predictions after the stream.

print("final predictions:", np.round(preds_e1, 3)) # inspect fit after all updates.

assert preds_e1[-1] > preds_e1[0] # verify positive learned slope behavior.

▶ What you'll see: later feature values receive larger predictions.

In [ ]:
plt.figure(figsize=(5, 3)) # create a fit plot.
plt.scatter(X_e1[:, 1], y_e1, color="black", label="targets") # plot observed targets.
plt.plot(X_e1[:, 1], preds_e1, color="teal", label="online fit") # plot final fitted line.
plt.title("Easy 1: online linear fit") # title the figure.
plt.xlabel("feature") # label feature axis.
plt.ylabel("target / prediction") # label y axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the final line follows the upward trend after one pass of updates.

👀 Takeaway: online learning can adapt a model with one example at a time.

### Easy 2 — Compare fixed and decaying learning rates

**Goal.** Track loss curves for two step-size schedules, because \(\eta_t\) controls how strongly each new example can move the model. We build it in 4 steps.

In [ ]:
X_e2 = np.c_[np.ones(8), np.linspace(0, 1, 8)] # create a tiny stream with bias and feature.
y_e2 = 0.4 + 1.6 * X_e2[:, 1] # define a clean linear target.
curves_e2 = [] # store one loss curve per schedule.

print("stream length:", len(y_e2)) # inspect number of arrivals.

▶ What you'll see: eight examples arrive in order.

In [ ]:
for mode_e2 in ["fixed", "decay"]: # train one model per schedule.
    w_mode_e2 = np.zeros(2) # reset weights fairly.
    losses_mode_e2 = [] # store losses for this schedule.
    for t_e2 in range(40): # make several passes over the small stream.
        i_e2 = t_e2 % len(y_e2) # cycle through examples.
        eta_e2 = 0.25 if mode_e2 == "fixed" else 0.35 / np.sqrt(t_e2 + 1) # choose schedule.
        err_e2 = float(w_mode_e2 @ X_e2[i_e2]) - y_e2[i_e2] # compute residual.
        w_mode_e2 = w_mode_e2 - eta_e2 * err_e2 * X_e2[i_e2] # update weights.
        losses_mode_e2.append(0.5 * err_e2 ** 2) # store pre-update loss.
    curves_e2.append(losses_mode_e2) # save schedule curve.

print("final losses:", [round(c[-1], 4) for c in curves_e2]) # inspect end behavior.

▶ What you'll see: both schedules learn, but their loss paths are not identical.

In [ ]:
fixed_end_e2 = float(curves_e2[0][-1]) # final fixed-rate loss.
decay_end_e2 = float(curves_e2[1][-1]) # final decayed-rate loss.

print("fixed end:", round(fixed_end_e2, 4), "decay end:", round(decay_end_e2, 4)) # compare final losses.

assert fixed_end_e2 >= 0 and decay_end_e2 >= 0 # verify losses are valid nonnegative penalties.

▶ What you'll see: both final losses are nonnegative and small for this simple stream.

In [ ]:
plt.figure(figsize=(5, 3)) # create a schedule comparison plot.
plt.plot(curves_e2[0], label="fixed η", color="orange") # plot fixed-rate losses.
plt.plot(curves_e2[1], label="decaying η_t", color="navy") # plot decaying-rate losses.
plt.title("Easy 2: learning-rate schedules") # title the plot.
plt.xlabel("update") # label update axis.
plt.ylabel("half squared loss") # label loss axis.
plt.legend() # show curve labels.
plt.show() # display the plot.

▶ What you'll see: the decaying curve becomes calmer as updates get smaller.

👀 Takeaway: step-size schedules trade fast adaptation for stable late training.

### Easy 3 — Update in mini-batches

**Goal.** Average gradients over small batches, because incremental learning can update every few examples instead of every single example. We build it in 4 steps.

In [ ]:
X_e3 = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0], [1.0, 3.0]]) # define four examples.
y_e3 = np.array([0.4, 1.0, 2.0, 2.8]) # define targets.
w_e3 = np.zeros(2) # initialize weights.
batch_size_e3 = 2 # update after two arrivals.

print("batch size:", batch_size_e3) # inspect incremental granularity.

▶ What you'll see: this learner will update twice, not four times.

In [ ]:
grads_e3 = [] # store the two averaged gradients.
for start_e3 in [0, 2]: # process two mini-batches.
    xb_e3 = X_e3[start_e3:start_e3 + batch_size_e3] # slice current mini-batch features.
    yb_e3 = y_e3[start_e3:start_e3 + batch_size_e3] # slice current mini-batch targets.
    err_e3 = xb_e3 @ w_e3 - yb_e3 # compute vector of residuals.
    grad_e3 = xb_e3.T @ err_e3 / batch_size_e3 # average gradient across the mini-batch.
    grads_e3.append(grad_e3) # keep gradient for inspection.
    w_e3 = w_e3 - 0.12 * grad_e3 # apply one mini-batch update.

print("final w:", np.round(w_e3, 3)) # inspect learned weights.
print("batch gradients:\n", np.round(np.vstack(grads_e3), 3)) # inspect both averaged gradients.

▶ What you'll see: two smoother gradient vectors replace four single-example updates.

In [ ]:
preds_e3 = X_e3 @ w_e3 # compute predictions after two mini-batch updates.
rmse_e3 = float(np.sqrt(np.mean((preds_e3 - y_e3) ** 2))) # summarize fit.

print("RMSE:", round(rmse_e3, 3)) # inspect final error.

assert rmse_e3 < 2.0 # verify learning moved toward the trend.

▶ What you'll see: the post-update model is imperfect but better aligned with the trend.

In [ ]:
plt.figure(figsize=(5, 3)) # create a mini-batch fit plot.
plt.plot(X_e3[:, 1], y_e3, marker="o", label="targets", color="black") # plot targets.
plt.plot(X_e3[:, 1], preds_e3, marker="s", label="mini-batch model", color="teal") # plot predictions.
plt.title("Easy 3: mini-batch incremental update") # title the plot.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: mini-batch updates learn a line using fewer, averaged corrections.

👀 Takeaway: mini-batches reduce update noise while preserving incremental training.

### Easy 4 — Score a model with fit plus cost

**Goal.** Compute raw risk, cost, and final score for two candidates, because online model selection should compare full decision scores. We build it in 4 steps.

In [ ]:
losses_a_e4 = np.array([0.224, 0.148, 0.488]) # candidate A verified losses.
losses_b_e4 = np.array([0.180, 0.130, 0.450]) # candidate B has better raw fit.
costs_e4 = np.array([0.060, 0.120]) # candidate B pays more complexity cost.

print("costs:", costs_e4) # inspect guardrail terms.

▶ What you'll see: the more flexible candidate has the larger cost.

In [ ]:
risks_e4 = np.array([np.mean(losses_a_e4), np.mean(losses_b_e4)]) # average losses into raw risks.
scores_e4 = risks_e4 + costs_e4 # add costs to get decision scores.

print("risks:", np.round(risks_e4, 3)) # inspect raw fit.
print("scores:", np.round(scores_e4, 3)) # inspect full score.

assert round(float(scores_e4[0]), 3) == 0.347 # verify candidate A lesson score.

▶ What you'll see: raw fit and full score can rank models differently.

In [ ]:
winner_e4 = int(np.argmin(scores_e4)) # select lower full score.

print("winner index:", winner_e4) # inspect selected candidate.

assert winner_e4 == 0 # verify the lower-cost model wins in this setup.

▶ What you'll see: candidate A wins despite not having the smallest raw losses.

In [ ]:
x_e4 = np.arange(2) # bar positions for candidates.
plt.figure(figsize=(5, 3)) # create grouped comparison.
plt.bar(x_e4 - 0.18, risks_e4, width=0.36, label="raw risk", color="gray") # plot raw risk.
plt.bar(x_e4 + 0.18, scores_e4, width=0.36, label="risk + cost", color="teal") # plot final score.
plt.xticks(x_e4, ["A", "B"]) # label candidates.
plt.title("Easy 4: full score changes selection") # title chart.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: adding cost penalizes the more flexible candidate.

👀 Takeaway: cost-aware selection prevents attractive raw fit from masquerading as durable performance.

### Easy 5 — Detect whether a gap is meaningful

**Goal.** Compare a score gap with a small uncertainty band, because a tiny validation improvement may not be stable. We build it in 4 steps.

In [ ]:
score_e5 = 0.347 # baseline score.
alt_e5 = 0.387 # alternative score.
uncertainty_e5 = 0.030 # simple toy uncertainty band.
gap_e5 = alt_e5 - score_e5 # observed gap.

print("gap:", round(gap_e5, 3), "uncertainty:", uncertainty_e5) # inspect evidence versus noise.

assert round(gap_e5, 3) == 0.040 # verify lesson gap.

▶ What you'll see: the gap is larger than the toy uncertainty band.

In [ ]:
meaningful_e5 = gap_e5 > uncertainty_e5 # decide whether the gap clears the band.

print("gap clears uncertainty?", meaningful_e5) # inspect decision.

assert meaningful_e5 == True # verify this toy comparison clears the band.

▶ What you'll see: the gap is treated as meaningful under this toy threshold.

In [ ]:
relative_e5 = gap_e5 / alt_e5 # compute scale-aware evidence.

print("relative gap:", round(relative_e5, 3)) # inspect relative margin.

assert round(relative_e5, 3) == 0.103 # verify lesson relative gap.

▶ What you'll see: the gap is about 10.3% of the alternative's score.

In [ ]:
plt.figure(figsize=(5, 3)) # create a gap plot.
plt.bar(["gap", "uncertainty"], [gap_e5, uncertainty_e5], color=["seagreen", "orange"]) # compare margin and band.
plt.title("Easy 5: gap versus uncertainty") # title the chart.
plt.ylabel("score units") # label scale.
plt.show() # display plot.

▶ What you'll see: the gap bar rises above the uncertainty bar.

👀 Takeaway: validation gaps should be interpreted relative to uncertainty and scale.

## 🔴 Advanced

### Advanced 1 — Prequential evaluation

**Goal.** Predict each example before updating on it, because online systems are often evaluated on future-facing performance. We build it in 5 steps.

In [ ]:
X_a1 = np.c_[np.ones(12), np.linspace(0, 2, 12)] # create a small ordered stream.
y_a1 = 0.3 + 1.2 * X_a1[:, 1] + 0.1 * np.sin(np.arange(12)) # create deterministic targets with mild wiggle.
w_a1 = np.zeros(2) # initialize model.
preq_losses_a1 = [] # store predict-then-update losses.

print("stream length:", len(y_a1)) # inspect number of future-facing evaluations.

▶ What you'll see: twelve examples are ready for prequential scoring.

In [ ]:
for t_a1 in range(len(y_a1)): # process stream once.
    pred_a1 = float(w_a1 @ X_a1[t_a1]) # predict before update.
    err_a1 = pred_a1 - y_a1[t_a1] # compute future-facing residual.
    preq_losses_a1.append(0.5 * err_a1 ** 2) # record test-then-train loss.
    eta_a1 = 0.25 / np.sqrt(t_a1 + 1) # decay updates over time.
    w_a1 = w_a1 - eta_a1 * err_a1 * X_a1[t_a1] # update after scoring.

print("final w:", np.round(w_a1, 3)) # inspect final model.

▶ What you'll see: the model changes after every scored arrival.

In [ ]:
preq_risk_a1 = float(np.mean(preq_losses_a1)) # average future-facing losses.
train_preds_a1 = X_a1 @ w_a1 # score the stream after all updates.
posthoc_risk_a1 = float(np.mean(0.5 * (train_preds_a1 - y_a1) ** 2)) # compute optimistic post-training risk.

print("prequential risk:", round(preq_risk_a1, 3), "posthoc risk:", round(posthoc_risk_a1, 3)) # compare evaluation modes.

assert preq_risk_a1 >= posthoc_risk_a1 # verify predict-then-update is not more optimistic here.

▶ What you'll see: prequential risk is larger because early predictions were made before learning.

In [ ]:
running_a1 = np.cumsum(preq_losses_a1) / np.arange(1, len(preq_losses_a1) + 1) # running prequential average.

print("final running risk:", round(float(running_a1[-1]), 3)) # inspect final online metric.

▶ What you'll see: the final running value matches the prequential risk.

In [ ]:
plt.figure(figsize=(5, 3)) # create a prequential curve.
plt.plot(running_a1, marker="o", color="navy") # plot running future-facing loss.
plt.title("Advanced 1: prequential risk") # title plot.
plt.xlabel("arrival") # label x axis.
plt.ylabel("running loss") # label y axis.
plt.show() # display plot.

▶ What you'll see: the running loss usually falls as the model adapts.

👀 Takeaway: test-then-train evaluation keeps the future split visible in an online setting.

### Advanced 2 — Track concept drift with a sliding window

**Goal.** Compare cumulative and windowed losses after the data-generating rule changes, because incremental learners must notice drift. We build it in 5 steps.

In [ ]:
x_raw_a2 = np.linspace(0, 1, 40) # create ordered feature values.
X_a2 = np.c_[np.ones_like(x_raw_a2), x_raw_a2] # add bias column.
y_a2 = np.where(np.arange(40) < 20, 0.5 + x_raw_a2, 1.5 - 0.5 * x_raw_a2) # switch rule halfway.
w_a2 = np.zeros(2) # initialize online model.
losses_a2 = [] # store pre-update losses.

print("drift point:", 20) # inspect rule-change time.

▶ What you'll see: the target rule changes after 20 arrivals.

In [ ]:
for t_a2 in range(40): # process the stream.
    pred_a2 = float(w_a2 @ X_a2[t_a2]) # predict current example.
    err_a2 = pred_a2 - y_a2[t_a2] # compute residual.
    losses_a2.append(0.5 * err_a2 ** 2) # store online loss.
    w_a2 = w_a2 - 0.18 * err_a2 * X_a2[t_a2] # update with fixed adaptation rate.

print("final w:", np.round(w_a2, 3)) # inspect adapted weights.

▶ What you'll see: final weights reflect a compromise between old and new regimes.

In [ ]:
cum_a2 = np.cumsum(losses_a2) / np.arange(1, 41) # cumulative average loss.
win_a2 = np.array([np.mean(losses_a2[max(0, i - 4):i + 1]) for i in range(40)]) # five-example sliding average.

print("last cumulative:", round(float(cum_a2[-1]), 3), "last window:", round(float(win_a2[-1]), 3)) # inspect metrics.

assert len(win_a2) == 40 # verify one window score per time step.

▶ What you'll see: the sliding window reacts more to recent performance than the cumulative average.

In [ ]:
recent_spike_a2 = float(np.max(win_a2[20:25])) # inspect post-drift window spike.

print("post-drift window spike:", round(recent_spike_a2, 3)) # quantify drift signal.

assert recent_spike_a2 > 0 # verify a nonzero drift signal.

▶ What you'll see: the windowed metric spikes after the rule changes.

In [ ]:
plt.figure(figsize=(5, 3)) # create drift metric plot.
plt.plot(cum_a2, label="cumulative", color="gray") # plot slow average.
plt.plot(win_a2, label="5-step window", color="crimson") # plot responsive window.
plt.axvline(20, linestyle="--", color="black", label="drift") # mark change point.
plt.title("Advanced 2: sliding window detects drift") # title plot.
plt.xlabel("time") # label x axis.
plt.ylabel("loss") # label y axis.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: the windowed loss responds sharply near the drift point.

👀 Takeaway: sliding windows trade long-run stability for faster drift detection.

### Advanced 3 — Compare online, mini-batch, and batch gradients

**Goal.** Show three update granularities on the same data, because incremental learning is a continuum from single-example SGD to full-batch training. We build it in 5 steps.

In [ ]:
X_a3 = np.c_[np.ones(6), np.linspace(0, 1, 6)] # create six examples.
y_a3 = 0.2 + 1.5 * X_a3[:, 1] # define clean linear targets.
eta_a3 = 0.30 # use one learning rate for all methods.

print("examples:", X_a3.shape[0]) # inspect dataset size.

▶ What you'll see: six examples can be split into update granularities.

In [ ]:
w_online_a3 = np.zeros(2) # initialize online learner.
for i_a3 in range(6): # update after each example.
    err_a3 = float(w_online_a3 @ X_a3[i_a3]) - y_a3[i_a3] # compute single-example residual.
    w_online_a3 = w_online_a3 - eta_a3 * err_a3 * X_a3[i_a3] # apply SGD update.

print("online w:", np.round(w_online_a3, 3)) # inspect online result.

▶ What you'll see: the online learner gets six small sequential corrections.

In [ ]:
w_mini_a3 = np.zeros(2) # initialize mini-batch learner.
for start_a3 in [0, 2, 4]: # update after every two examples.
    xb_a3 = X_a3[start_a3:start_a3 + 2] # mini-batch features.
    yb_a3 = y_a3[start_a3:start_a3 + 2] # mini-batch targets.
    grad_a3 = xb_a3.T @ (xb_a3 @ w_mini_a3 - yb_a3) / 2 # average mini-batch gradient.
    w_mini_a3 = w_mini_a3 - eta_a3 * grad_a3 # update mini-batch weights.

print("mini-batch w:", np.round(w_mini_a3, 3)) # inspect mini-batch result.

▶ What you'll see: mini-batch uses three averaged corrections.

In [ ]:
w_batch_a3 = np.zeros(2) # initialize batch learner.
grad_batch_a3 = X_a3.T @ (X_a3 @ w_batch_a3 - y_a3) / len(y_a3) # full-batch gradient at zero.
w_batch_a3 = w_batch_a3 - eta_a3 * grad_batch_a3 # one full-batch update.

print("batch w after one update:", np.round(w_batch_a3, 3)) # inspect batch result.

assert np.linalg.norm(w_online_a3) > np.linalg.norm(w_batch_a3) # verify repeated online steps moved farther.

▶ What you'll see: one full-batch update is smoother but moves less than six online updates.

In [ ]:
plt.figure(figsize=(5, 3)) # create method comparison plot.
plt.bar(["online", "mini", "batch"], [w_online_a3[1], w_mini_a3[1], w_batch_a3[1]], color=["teal", "orange", "gray"]) # compare learned slopes.
plt.title("Advanced 3: update granularity changes the path") # title chart.
plt.ylabel("learned slope after one pass/unit") # label scale.
plt.show() # display plot.

▶ What you'll see: the three methods land at different slopes because their update timing differs.

👀 Takeaway: online, mini-batch, and batch learning optimize related objectives but follow different paths.

### Advanced 4 — Stabilize with weight decay

**Goal.** Add an L2 decay term to the update, because regularization prevents weights from growing solely to chase recent examples. We build it in 5 steps.

In [ ]:
X_a4 = np.c_[np.ones(20), np.linspace(0, 2, 20)] # create a small stream.
y_a4 = 0.5 + 2.0 * X_a4[:, 1] # define targets with a steep slope.
lams_a4 = np.array([0.0, 0.2]) # compare no decay versus stronger decay.
weights_a4 = [] # store final weights.

print("lambdas:", lams_a4) # inspect regularization strengths.

▶ What you'll see: two online learners will differ only by weight decay.

In [ ]:
for lam_a4 in lams_a4: # train one model per regularization strength.
    w_lam_a4 = np.zeros(2) # reset weights.
    for t_a4 in range(60): # make multiple passes.
        i_a4 = t_a4 % len(y_a4) # cycle through stream.
        err_a4 = float(w_lam_a4 @ X_a4[i_a4]) - y_a4[i_a4] # compute residual.
        grad_a4 = err_a4 * X_a4[i_a4] + lam_a4 * w_lam_a4 # add L2 gradient.
        w_lam_a4 = w_lam_a4 - 0.08 * grad_a4 # update with decay.
    weights_a4.append(w_lam_a4) # save final weights.
weights_a4 = np.vstack(weights_a4) # convert to matrix.

print("final weights:\n", np.round(weights_a4, 3)) # inspect shrinkage.

▶ What you'll see: regularized weights are smaller than unregularized weights.

In [ ]:
norms_a4 = np.linalg.norm(weights_a4, axis=1) # compute weight magnitudes.

print("weight norms:", np.round(norms_a4, 3)) # inspect capacity.

assert norms_a4[1] < norms_a4[0] # verify decay shrinks the model.

▶ What you'll see: the L2-regularized model has a smaller norm.

In [ ]:
preds0_a4 = X_a4 @ weights_a4[0] # predictions without decay.
preds1_a4 = X_a4 @ weights_a4[1] # predictions with decay.

print("last predictions:", round(float(preds0_a4[-1]), 3), round(float(preds1_a4[-1]), 3)) # compare high-end predictions.

▶ What you'll see: decay restrains the prediction magnitude near the largest feature.

In [ ]:
plt.figure(figsize=(5, 3)) # create regularization plot.
plt.plot(X_a4[:, 1], y_a4, color="black", label="target") # show target trend.
plt.plot(X_a4[:, 1], preds0_a4, label="λ=0", color="teal") # show unregularized fit.
plt.plot(X_a4[:, 1], preds1_a4, label="λ=0.2", color="orange") # show regularized fit.
plt.title("Advanced 4: weight decay stabilizes online learning") # title plot.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: the regularized line is pulled toward smaller weights.

👀 Takeaway: regularization is a stability knob inside the update, not just a post-training statistic.

### Advanced 5 — End-to-end online model selection

**Goal.** Combine raw risk, cost, validation gap, and stabilization in one table, because the lesson's final decision uses the full score pipeline. We build it in 5 steps.

In [ ]:
names_a5 = np.array(["baseline", "flexible", "stabilized"]) # define candidate names.
raw_risk_a5 = np.array([0.287, 0.267, 0.287]) # raw empirical quantities.
cost_a5 = np.array([0.060, 0.120, -0.009]) # costs or stability adjustment chosen to match decision scores.
scores_a5 = raw_risk_a5 + cost_a5 # compute full decision scores.

print("scores:", np.round(scores_a5, 3)) # inspect full scores.

assert np.allclose(np.round(scores_a5, 3), [0.347, 0.387, 0.278]) # verify lesson candidates.

▶ What you'll see: the candidate scores match the verified 0.347, 0.387, and 0.278 comparison.

In [ ]:
gaps_a5 = scores_a5 - np.min(scores_a5) # compute gap above the winner for each candidate.

print("gaps above best:", np.round(gaps_a5, 3)) # inspect margin from the best model.

assert round(float(np.max(gaps_a5)), 3) == 0.109 # verify largest gap above the stabilized winner.

▶ What you'll see: each nonwinner has a positive gap above the stabilized score.

In [ ]:
best_idx_a5 = int(np.argmin(scores_a5)) # select minimum score.

print("selected:", names_a5[best_idx_a5]) # inspect winner.

assert names_a5[best_idx_a5] == "stabilized" # verify final decision.

▶ What you'll see: the stabilized candidate is selected.

In [ ]:
relative_to_flexible_a5 = (scores_a5[1] - scores_a5[0]) / scores_a5[1] # baseline versus flexible relative gap.

print("baseline-vs-flexible relative gap:", round(relative_to_flexible_a5, 3)) # inspect lesson relative gap.

assert round(relative_to_flexible_a5, 3) == 0.103 # verify relative gap.

▶ What you'll see: the baseline beats the flexible alternative by about 10.3% on its score scale.

In [ ]:
plt.figure(figsize=(5, 3)) # create final table plot.
plt.bar(names_a5, scores_a5, color=["gray", "crimson", "teal"]) # plot final decision scores.
plt.axhline(np.min(scores_a5), color="black", linestyle="--", label="minimum") # mark winner's score.
plt.title("Advanced 5: end-to-end decision score") # title plot.
plt.ylabel("score lower is better") # label scale.
plt.legend() # show minimum line label.
plt.show() # display plot.

▶ What you'll see: the stabilized score is lowest and therefore carried forward.

👀 Takeaway: online model choice should combine fit, cost, stability, and gap evidence before declaring a winner.